In [827]:

# SEAG qualification period is 22 Oct 2024 to 5 Sept 2025

#1. Segregate into Male and Femal 
#2. For each gender perform the following: 
#a. Sort data by mapped eent, then perf scalar (higher the better)
#b. Identify tiers based on performance - Tier 1 is meets bronze medal mark for SEAG, Tier 2 is 2% and Tier 3 is 3.5%
#c. Check - if athlete met bronze or 2%/3.5% then delta_benchmark is zero or +, delta2% is + and delta 3.5% is +
#d. Top ranked athletes for each event are chosen. Max number of athletes for each event is 3, except for 100m/400m which is 6
#    This includes athletes on spex scholarship and potential
#e. The max for each tier is 2. Lower ranked athletes move down one tier.
#3. If athlete qualifies for more than one event the higher tier event is given
#4. Jump and throws junior program to be solved separately

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [828]:
# Import usual modules
import pandas as pd
import csv
import math
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import openpyxl
import datetime
from scipy.stats import lognorm
import re
import string
from bs4 import BeautifulSoup
import requests
import unicodedata # for removing accented characters
import datetime
import icecream as ic
import dateutil.parser as parser 
import datacompy
import pytz
import gspread

from google.cloud import storage



In [829]:
# PRODUCTION ENVIRONMENT
# Extract timed event records

import pandas_gbq
from google.oauth2 import service_account

credentials = service_account.Credentials.from_service_account_file(
    '/Users/veesheenyuen/Desktop/DataScience/Keys/saa-analytics-7c8937b70609.json',
    
    
)

sql1="""
SELECT NAME, RESULT, TEAM, AGE, RANK AS COMPETITION_RANK, DIVISION, EVENT, DISTANCE, EVENT_CLASS, UNIQUE_ID, DOB, NATIONALITY, WIND, CATEGORY_EVENT, GENDER, COMPETITION, DATE, YEAR, REGION, TIMESTAMP
FROM `saa-analytics.results.PRODUCTION_CORRECT` 
WHERE CATEGORY_EVENT='Jump' AND RESULT!='NM' AND RESULT!='-' AND RESULT!='DNS' AND RESULT!='DNF' AND RESULT!='DNQ' AND RESULT!='DQ'  AND RESULT!='FOUL' AND RESULT IS NOT NULL

"""

competitors = pandas_gbq.read_gbq(sql1, project_id="saa-analytics", credentials=credentials)




Downloading: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████|


In [830]:
competitors

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,DOB,NATIONALITY,WIND,CATEGORY_EVENT,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP
0,"Sim, Jarod",4.98m,Raffles Institution,15,20,U18,Long Jump,0.0,None,J727I11,2011-03-29,sin,+0.0,Jump,Male,SA All Comers Meet 2,2026-02-08 00:00:00+00:00,2026,Local,2026-02-11 16:14:29.851514
1,"Adam, Rabin Sunder Singh",4.83m,Raffles Institution,15,24,U18,Long Jump,0.0,None,R762W11,2011-04-17,sin,-0.2,Jump,Male,SA All Comers Meet 2,2026-02-08 00:00:00+00:00,2026,Local,2026-02-11 16:14:29.851514
2,"Chua, Hoon Kai",4.61m,Raffles Institution,15,34,U18,Long Jump,0.0,None,H269F11,2011-02-27,sin,+0.0,Jump,Male,SA All Comers Meet 2,2026-02-08 00:00:00+00:00,2026,Local,2026-02-11 16:14:29.851514
3,"NEALFAIZ ADAM, MOHAMAD NAZRUL NAZIB",4.73m,Beecity Athletics Academy,17,27,U18,Long Jump,0.0,None,M209309,2009-01-23,mas,-0.6,Jump,Male,SA All Comers Meet 2,2026-02-08 00:00:00+00:00,2026,Local,2026-02-11 16:14:29.851514
4,"TANG SONG YI, Zachary",5.25m,St Joseph's Institution,15,9,U18,Long Jump,0.0,None,None,2011-03-16,None,+0.0,Jump,Male,SA All Comers Meet 2,2026-02-08 00:00:00+00:00,2026,Local,2026-02-11 16:14:29.851514
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21556,"CHENG, ZIXUAN",1.40m,Catholic High School,14,5,Novice,High Jump,0.0,None,Z573D12,2012-04-11,sin,None,Jump,Male,SA All Comers Meet 1,2026-02-01 00:00:00+00:00,2026,Local,2026-02-07 13:23:03.305409
21557,"Ho, Yi Teng",1.46m,Hwa Chong Institution,14,2,Novice,High Jump,0.0,None,D372C12,2012-04-03,sin,None,Jump,Male,SA All Comers Meet 1,2026-02-01 00:00:00+00:00,2026,Local,2026-02-07 13:23:03.305409
21558,"Boon, Marius",1.40m,Hwa Chong Institution,14,5,Novice,High Jump,0.0,None,Z430G12,2012-05-26,sin,None,Jump,Male,SA All Comers Meet 1,2026-02-01 00:00:00+00:00,2026,Local,2026-02-07 13:23:03.305409
21559,"Tan, Ashton",1.80m,Hwa Chong Alumni Association,21,1,Advance,High Jump,0.0,None,A401I05,2005-03-30,sin,None,Jump,Male,SA All Comers Meet 1,2026-02-01 00:00:00+00:00,2026,Local,2026-02-07 13:23:03.305409


In [831]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Jumps/')


competitors.to_csv('database_jumps.csv', sep=',', encoding='utf-8-sig', index=False)

In [832]:
competitors

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,DOB,NATIONALITY,WIND,CATEGORY_EVENT,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP
0,"Sim, Jarod",4.98m,Raffles Institution,15,20,U18,Long Jump,0.0,None,J727I11,2011-03-29,sin,+0.0,Jump,Male,SA All Comers Meet 2,2026-02-08 00:00:00+00:00,2026,Local,2026-02-11 16:14:29.851514
1,"Adam, Rabin Sunder Singh",4.83m,Raffles Institution,15,24,U18,Long Jump,0.0,None,R762W11,2011-04-17,sin,-0.2,Jump,Male,SA All Comers Meet 2,2026-02-08 00:00:00+00:00,2026,Local,2026-02-11 16:14:29.851514
2,"Chua, Hoon Kai",4.61m,Raffles Institution,15,34,U18,Long Jump,0.0,None,H269F11,2011-02-27,sin,+0.0,Jump,Male,SA All Comers Meet 2,2026-02-08 00:00:00+00:00,2026,Local,2026-02-11 16:14:29.851514
3,"NEALFAIZ ADAM, MOHAMAD NAZRUL NAZIB",4.73m,Beecity Athletics Academy,17,27,U18,Long Jump,0.0,None,M209309,2009-01-23,mas,-0.6,Jump,Male,SA All Comers Meet 2,2026-02-08 00:00:00+00:00,2026,Local,2026-02-11 16:14:29.851514
4,"TANG SONG YI, Zachary",5.25m,St Joseph's Institution,15,9,U18,Long Jump,0.0,None,None,2011-03-16,None,+0.0,Jump,Male,SA All Comers Meet 2,2026-02-08 00:00:00+00:00,2026,Local,2026-02-11 16:14:29.851514
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21556,"CHENG, ZIXUAN",1.40m,Catholic High School,14,5,Novice,High Jump,0.0,None,Z573D12,2012-04-11,sin,None,Jump,Male,SA All Comers Meet 1,2026-02-01 00:00:00+00:00,2026,Local,2026-02-07 13:23:03.305409
21557,"Ho, Yi Teng",1.46m,Hwa Chong Institution,14,2,Novice,High Jump,0.0,None,D372C12,2012-04-03,sin,None,Jump,Male,SA All Comers Meet 1,2026-02-01 00:00:00+00:00,2026,Local,2026-02-07 13:23:03.305409
21558,"Boon, Marius",1.40m,Hwa Chong Institution,14,5,Novice,High Jump,0.0,None,Z430G12,2012-05-26,sin,None,Jump,Male,SA All Comers Meet 1,2026-02-01 00:00:00+00:00,2026,Local,2026-02-07 13:23:03.305409
21559,"Tan, Ashton",1.80m,Hwa Chong Alumni Association,21,1,Advance,High Jump,0.0,None,A401I05,2005-03-30,sin,None,Jump,Male,SA All Comers Meet 1,2026-02-01 00:00:00+00:00,2026,Local,2026-02-07 13:23:03.305409


In [833]:
# DATE column to contain timezone - tz aware mode

competitors['DATE'] = pd.to_datetime(competitors['DATE'], format='mixed', dayfirst=False, utc=True)


In [834]:
# datetime to contain UTC (timezone)

competitors['NOW'] = datetime.datetime.now()

timezone = pytz.timezone('UTC')

competitors['NOW'] = datetime.datetime.now().replace(tzinfo=timezone)

In [835]:
# Calculate number of days from today to event date

#competitors['DATE'] = pd.to_datetime(competitors['DATE'], format='mixed', dayfirst=False, utc=False)

competitors['delta_time'] = competitors['NOW'] - competitors['DATE']


#competitors['delta_time'] = datetime.datetime.now() - competitors['DATE']


competitors['delta_time_conv'] = pd.to_numeric(competitors['delta_time'].dt.days, downcast='integer')

competitors['event_month'] = competitors['DATE'].dt.month

# Make sure date conversion is is valid for all rows

assert not competitors['delta_time'].isna().any()

In [836]:
# Choose date range for SEAG qualification window from Oct 22 to current


#competitors = competitors[(competitors['delta_time_conv']>=0) & (competitors['delta_time_conv']<=365)]

#competitors=competitors.reset_index(drop=True)

competitors['DATE']=competitors['DATE'].dt.tz_localize(None)  # switch off timezone for compatibility with np.datetime64


#start = datetime.datetime(2024, 1, 2)
start = datetime.datetime(2025, 1, 1)


end = datetime.datetime(2026, 2, 9)

start_date = np.datetime64(start)
end_date = np.datetime64(end)


mask = (competitors['DATE'] >= start_date) & (competitors['DATE'] <= end_date)
athletes_selected = competitors.loc[mask]



In [837]:
end_date - start_date

numpy.timedelta64(34905600000000,'us')

In [838]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Jumps/')


athletes_selected.to_csv('athletes_selected.csv', encoding='utf-8')

In [839]:
athletes_selected

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month
0,"Sim, Jarod",4.98m,Raffles Institution,15,20,U18,Long Jump,0.0,None,J727I11,...,Male,SA All Comers Meet 2,2026-02-08,2026,Local,2026-02-11 16:14:29.851514,2026-02-11 16:22:17.003853+00:00,3 days 16:22:17.003853,3,2
1,"Adam, Rabin Sunder Singh",4.83m,Raffles Institution,15,24,U18,Long Jump,0.0,None,R762W11,...,Male,SA All Comers Meet 2,2026-02-08,2026,Local,2026-02-11 16:14:29.851514,2026-02-11 16:22:17.003853+00:00,3 days 16:22:17.003853,3,2
2,"Chua, Hoon Kai",4.61m,Raffles Institution,15,34,U18,Long Jump,0.0,None,H269F11,...,Male,SA All Comers Meet 2,2026-02-08,2026,Local,2026-02-11 16:14:29.851514,2026-02-11 16:22:17.003853+00:00,3 days 16:22:17.003853,3,2
3,"NEALFAIZ ADAM, MOHAMAD NAZRUL NAZIB",4.73m,Beecity Athletics Academy,17,27,U18,Long Jump,0.0,None,M209309,...,Male,SA All Comers Meet 2,2026-02-08,2026,Local,2026-02-11 16:14:29.851514,2026-02-11 16:22:17.003853+00:00,3 days 16:22:17.003853,3,2
4,"TANG SONG YI, Zachary",5.25m,St Joseph's Institution,15,9,U18,Long Jump,0.0,None,None,...,Male,SA All Comers Meet 2,2026-02-08,2026,Local,2026-02-11 16:14:29.851514,2026-02-11 16:22:17.003853+00:00,3 days 16:22:17.003853,3,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21556,"CHENG, ZIXUAN",1.40m,Catholic High School,14,5,Novice,High Jump,0.0,None,Z573D12,...,Male,SA All Comers Meet 1,2026-02-01,2026,Local,2026-02-07 13:23:03.305409,2026-02-11 16:22:17.003853+00:00,10 days 16:22:17.003853,10,2
21557,"Ho, Yi Teng",1.46m,Hwa Chong Institution,14,2,Novice,High Jump,0.0,None,D372C12,...,Male,SA All Comers Meet 1,2026-02-01,2026,Local,2026-02-07 13:23:03.305409,2026-02-11 16:22:17.003853+00:00,10 days 16:22:17.003853,10,2
21558,"Boon, Marius",1.40m,Hwa Chong Institution,14,5,Novice,High Jump,0.0,None,Z430G12,...,Male,SA All Comers Meet 1,2026-02-01,2026,Local,2026-02-07 13:23:03.305409,2026-02-11 16:22:17.003853+00:00,10 days 16:22:17.003853,10,2
21559,"Tan, Ashton",1.80m,Hwa Chong Alumni Association,21,1,Advance,High Jump,0.0,None,A401I05,...,Male,SA All Comers Meet 1,2026-02-01,2026,Local,2026-02-07 13:23:03.305409,2026-02-11 16:22:17.003853+00:00,10 days 16:22:17.003853,10,2


In [840]:
# Choose 2024/25 only

athletes = athletes_selected

In [841]:
athletes

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month
0,"Sim, Jarod",4.98m,Raffles Institution,15,20,U18,Long Jump,0.0,None,J727I11,...,Male,SA All Comers Meet 2,2026-02-08,2026,Local,2026-02-11 16:14:29.851514,2026-02-11 16:22:17.003853+00:00,3 days 16:22:17.003853,3,2
1,"Adam, Rabin Sunder Singh",4.83m,Raffles Institution,15,24,U18,Long Jump,0.0,None,R762W11,...,Male,SA All Comers Meet 2,2026-02-08,2026,Local,2026-02-11 16:14:29.851514,2026-02-11 16:22:17.003853+00:00,3 days 16:22:17.003853,3,2
2,"Chua, Hoon Kai",4.61m,Raffles Institution,15,34,U18,Long Jump,0.0,None,H269F11,...,Male,SA All Comers Meet 2,2026-02-08,2026,Local,2026-02-11 16:14:29.851514,2026-02-11 16:22:17.003853+00:00,3 days 16:22:17.003853,3,2
3,"NEALFAIZ ADAM, MOHAMAD NAZRUL NAZIB",4.73m,Beecity Athletics Academy,17,27,U18,Long Jump,0.0,None,M209309,...,Male,SA All Comers Meet 2,2026-02-08,2026,Local,2026-02-11 16:14:29.851514,2026-02-11 16:22:17.003853+00:00,3 days 16:22:17.003853,3,2
4,"TANG SONG YI, Zachary",5.25m,St Joseph's Institution,15,9,U18,Long Jump,0.0,None,None,...,Male,SA All Comers Meet 2,2026-02-08,2026,Local,2026-02-11 16:14:29.851514,2026-02-11 16:22:17.003853+00:00,3 days 16:22:17.003853,3,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21556,"CHENG, ZIXUAN",1.40m,Catholic High School,14,5,Novice,High Jump,0.0,None,Z573D12,...,Male,SA All Comers Meet 1,2026-02-01,2026,Local,2026-02-07 13:23:03.305409,2026-02-11 16:22:17.003853+00:00,10 days 16:22:17.003853,10,2
21557,"Ho, Yi Teng",1.46m,Hwa Chong Institution,14,2,Novice,High Jump,0.0,None,D372C12,...,Male,SA All Comers Meet 1,2026-02-01,2026,Local,2026-02-07 13:23:03.305409,2026-02-11 16:22:17.003853+00:00,10 days 16:22:17.003853,10,2
21558,"Boon, Marius",1.40m,Hwa Chong Institution,14,5,Novice,High Jump,0.0,None,Z430G12,...,Male,SA All Comers Meet 1,2026-02-01,2026,Local,2026-02-07 13:23:03.305409,2026-02-11 16:22:17.003853+00:00,10 days 16:22:17.003853,10,2
21559,"Tan, Ashton",1.80m,Hwa Chong Alumni Association,21,1,Advance,High Jump,0.0,None,A401I05,...,Male,SA All Comers Meet 1,2026-02-01,2026,Local,2026-02-07 13:23:03.305409,2026-02-11 16:22:17.003853+00:00,10 days 16:22:17.003853,10,2


In [842]:
for col in athletes.columns:
    athletes[col] = athletes[col].astype(str)
    athletes[col] = athletes[col].str.replace('\xa0', ' ', regex=True)
    athletes[col] = athletes[col].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
    athletes[col] = athletes[col].str.replace('\r', ' ', regex=True)
    athletes[col] = athletes[col].str.replace('\n', ' ', regex=True)
    athletes[col] = athletes[col].str.strip()

/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_11839/2203170124.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes[col] = athletes[col].astype(str)
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_11839/2203170124.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes[col] = athletes[col].str.replace('\xa0', ' ', regex=True)
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_11839/2203170124.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of

/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_81269/2203170124.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes[col] = athletes[col].str.replace('\xa0', ' ', regex=True)
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_81269/2203170124.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes[col] = athletes[col].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_81269/2203170124.py:5: SettingWithCopyWarning: 


/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_81269/2203170124.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes[col] = athletes[col].str.strip()
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_81269/2203170124.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes[col] = athletes[col].astype(str)
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_81269/2203170124.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

In [843]:
athletes

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month
0,"Sim, Jarod",4.98m,Raffles Institution,15,20,U18,Long Jump,0.0,None,J727I11,...,Male,SA All Comers Meet 2,2026-02-08 00:00:00,2026,Local,2026-02-11 16:14:29.851514,2026-02-11 16:22:17.003853+00:00,3 days 16:22:17.003853,3,2
1,"Adam, Rabin Sunder Singh",4.83m,Raffles Institution,15,24,U18,Long Jump,0.0,None,R762W11,...,Male,SA All Comers Meet 2,2026-02-08 00:00:00,2026,Local,2026-02-11 16:14:29.851514,2026-02-11 16:22:17.003853+00:00,3 days 16:22:17.003853,3,2
2,"Chua, Hoon Kai",4.61m,Raffles Institution,15,34,U18,Long Jump,0.0,None,H269F11,...,Male,SA All Comers Meet 2,2026-02-08 00:00:00,2026,Local,2026-02-11 16:14:29.851514,2026-02-11 16:22:17.003853+00:00,3 days 16:22:17.003853,3,2
3,"NEALFAIZ ADAM, MOHAMAD NAZRUL NAZIB",4.73m,Beecity Athletics Academy,17,27,U18,Long Jump,0.0,None,M209309,...,Male,SA All Comers Meet 2,2026-02-08 00:00:00,2026,Local,2026-02-11 16:14:29.851514,2026-02-11 16:22:17.003853+00:00,3 days 16:22:17.003853,3,2
4,"TANG SONG YI, Zachary",5.25m,St Joseph's Institution,15,9,U18,Long Jump,0.0,None,None,...,Male,SA All Comers Meet 2,2026-02-08 00:00:00,2026,Local,2026-02-11 16:14:29.851514,2026-02-11 16:22:17.003853+00:00,3 days 16:22:17.003853,3,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21556,"CHENG, ZIXUAN",1.40m,Catholic High School,14,5,Novice,High Jump,0.0,None,Z573D12,...,Male,SA All Comers Meet 1,2026-02-01 00:00:00,2026,Local,2026-02-07 13:23:03.305409,2026-02-11 16:22:17.003853+00:00,10 days 16:22:17.003853,10,2
21557,"Ho, Yi Teng",1.46m,Hwa Chong Institution,14,2,Novice,High Jump,0.0,None,D372C12,...,Male,SA All Comers Meet 1,2026-02-01 00:00:00,2026,Local,2026-02-07 13:23:03.305409,2026-02-11 16:22:17.003853+00:00,10 days 16:22:17.003853,10,2
21558,"Boon, Marius",1.40m,Hwa Chong Institution,14,5,Novice,High Jump,0.0,None,Z430G12,...,Male,SA All Comers Meet 1,2026-02-01 00:00:00,2026,Local,2026-02-07 13:23:03.305409,2026-02-11 16:22:17.003853+00:00,10 days 16:22:17.003853,10,2
21559,"Tan, Ashton",1.80m,Hwa Chong Alumni Association,21,1,Advance,High Jump,0.0,None,A401I05,...,Male,SA All Comers Meet 1,2026-02-01 00:00:00,2026,Local,2026-02-07 13:23:03.305409,2026-02-11 16:22:17.003853+00:00,10 days 16:22:17.003853,10,2


In [844]:
# Read benchmarks file

os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Jumps/')

benchmarks = pd.read_csv('Benchmarks.csv')


In [845]:
benchmarks

,GENDER,SQUAD,EVENT,BENCHMARK
0,Male,NATIONAL,Long Jump,7.20
1,Female,NATIONAL,Long Jump,5.70
2,Male,NATIONAL,Triple Jump,15.20
3,Female,NATIONAL,Triple Jump,12.00
4,Male,NATIONAL,High Jump,2.07
5,Female,NATIONAL,High Jump,1.65
6,Male,TRAINING,Long Jump,6.80
7,Female,TRAINING,Long Jump,5.30
8,Male,TRAINING,Triple Jump,14.50
9,Female,TRAINING,Triple Jump,11.30


In [846]:
# Handles 00:XX.XX format and converts into XX.XX

def convert_time_refactored(i, string, metric):
    """
    Convert various metric formats (distance, time) to a float value (primarily seconds for times).
    Optimized for speed: no global variables, no print statements, no unnecessary conversions.
    
    Args:
        i (int): Index (unused, kept for compatibility).
        string (str): Event description.
        metric (str, float, or datetime): The result metric.
    
    Returns:
        float or empty string: Converted metric as float (seconds/meters), or '' if not convertible.
    """
    l = ['discus', 'throw', 'jump', 'vault', 'shot']
    sprint_events = ['100m', '200m', '400m']
    
    string = str(string).lower()
    metric_str = str(metric)
    output = ''
    
    try:
        # Skip marks with illegal wind speeds
        if isinstance(metric_str, str) and 'w' in metric_str.lower():
            return ''
        
        # Field events (distances)
        if any(s in string for s in l):
            # Remove unit if present
            metric_clean = metric_str.replace('m', '').replace('GR', '')
            return round(float(metric_clean), 2)
        
        # No event description
        if string == '':
            return ''
        
        # Time events
        count_colon = metric_str.count(':')
        count_dot = metric_str.count('.')
        
        # Simple time as float (no colon)
        if count_colon == 0:
            return round(float(metric_str), 2)
        
        # Sprint events (100m, 200m, 400m): Handle 00:MM.SS or MM.SS format as seconds
        if any(sprint in string for sprint in sprint_events):
            if count_colon == 1 and count_dot == 1:
                # Format: 00:09.16 or 09.16
                parts = metric_str.split(':')
                if len(parts) == 2:
                    # Check if first part is "00" (ignore it) or actual minutes
                    first_part = parts[0]
                    second_part = parts[1]
                    
                    if first_part == '00':
                        # It's 00:SS.ss format, just return the seconds part
                        return float(second_part)
                    else:
                        # It's MM:SS.ss format, convert normally
                        return float(int(first_part) * 60 + float(second_part))
        
        # Convert time formats with two colons (like XX:XX:XX, XX:XX.XX)
        if count_colon == 2:

            # Check if this is a standard HH:MM:SS (no decimal point)
            if count_dot == 0:
                h, m, s = metric_str.split(':')
                return float(
                    int(h) * 3600 + int(m) * 60 + float(s)
                )
            # For 10,000m, 5000m, and 1500m, replace the 6th character with '.' for format XX:XX.XX
            if ('10,000m' in string or '5000m' in string or '1500m' in string):
                if len(metric_str) == 7:  # X:XX:XX (1500m special case)
                    idx = 4
                    metric_mod = '0' + metric_str[:idx] + '.' + metric_str[idx+1:]
                else:
                    idx = 5
                    metric_mod = metric_str[:idx] + '.' + metric_str[idx+1:]
                m, s = metric_mod.split(':')[-2:]
                return float((int(m) * 60) + float(s))
            
            # Standard HH:MM:SS
            h, m, s = metric_str.split(':')
            return float(int(h) * 3600 + int(m) * 60 + float(s))
        
        # Handle datetime.time/datetime.datetime objects
        if isinstance(metric, (datetime.time, datetime.datetime)):
            t = str(metric)
            h, m, s = t.split(':')
            return float(int(h) * 3600 + int(m) * 60 + float(s))
        
        # MM:SS.sss format
        if count_colon == 1 and count_dot >= 1:
            m, s = metric_str.split(':')
            return float(int(m) * 60 + float(s))
        
        # HH.MM.SS (rare) or MM:SS:SS
        if count_colon == 1 and count_dot == 2:
            # Replace first dot with colon
            metric_mod = metric_str.replace('.', ':', 1)
            h, m, s = metric_mod.split(':')
            return float(int(h) * 3600 + int(m) * 60 + float(s))
        
        # HH:MM:SS (no dots)
        if count_colon == 2 and count_dot == 0:
            h, m, s = metric_str.split(':')
            return float(int(h) * 3600 + int(m) * 60 + float(s))
        
        # MM:SS (no dots)
        if count_colon == 1 and count_dot == 0:
            m, s = metric_str.split(':')
            return float(int(m) * 60 + int(s))
            
    except Exception:
        return ''
    
    return output

In [847]:
def process_benchmarks(df):
    
    for i in range(len(df)):

        rowIndex = df.index[i]

        input_string=df.iloc[rowIndex,0]
    
        metric=df.iloc[rowIndex,3]
    
        if metric==None:
        
            continue
        
        out = convert_time_refactored(i, input_string, metric)
        
        print(rowIndex, input_string, out)

    
        df.loc[rowIndex, 'Metric'] = out
    
    return df

In [848]:
process_benchmarks(benchmarks)

0 Male 7.2
1 Female 5.7
2 Male 15.2
3 Female 12.0
4 Male 2.07
5 Female 1.65
6 Male 6.8
7 Female 5.3
8 Male 14.5
9 Female 11.3
10 Male 1.95
11 Female 1.57


,GENDER,SQUAD,EVENT,BENCHMARK,Metric
0,Male,NATIONAL,Long Jump,7.20,7.20
1,Female,NATIONAL,Long Jump,5.70,5.70
2,Male,NATIONAL,Triple Jump,15.20,15.20
3,Female,NATIONAL,Triple Jump,12.00,12.00
4,Male,NATIONAL,High Jump,2.07,2.07
5,Female,NATIONAL,High Jump,1.65,1.65
6,Male,TRAINING,Long Jump,6.80,6.80
7,Female,TRAINING,Long Jump,5.30,5.30
8,Male,TRAINING,Triple Jump,14.50,14.50
9,Female,TRAINING,Triple Jump,11.30,11.30


In [849]:
mask = benchmarks['EVENT'].str.lower().str.contains(r'jump|throw|put|pole|decathlon|heptathlon', na=True)

benchmarks.loc[mask, '2%']=benchmarks['BENCHMARK']*0.98
benchmarks.loc[mask, '3.5%']=benchmarks['BENCHMARK']*0.965
benchmarks.loc[mask, '5%']=benchmarks['BENCHMARK']*0.95
benchmarks.loc[mask, '10%']=benchmarks['BENCHMARK']*0.90


benchmarks.loc[~mask, '2%']=benchmarks['BENCHMARK']*1.02
benchmarks.loc[~mask, '3.5%']=benchmarks['BENCHMARK']*1.035
benchmarks.loc[~mask, '5%']=benchmarks['BENCHMARK']*1.05
benchmarks.loc[~mask, '10%']=benchmarks['BENCHMARK']*1.10


In [850]:
for col in benchmarks.columns:
    benchmarks[col] = benchmarks[col].astype(str)
    benchmarks[col] = benchmarks[col].str.replace('\xa0', ' ', regex=True)
    benchmarks[col] = benchmarks[col].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
    benchmarks[col] = benchmarks[col].str.replace('\r', ' ', regex=True)
    benchmarks[col] = benchmarks[col].str.replace('\n', ' ', regex=True)
    benchmarks[col] = benchmarks[col].str.strip()


In [851]:
benchmarks

,GENDER,SQUAD,EVENT,BENCHMARK,Metric,2%,3.5%,5%,10%
0,Male,NATIONAL,Long Jump,7.2,7.2,7.056,6.9479999999999995,6.84,6.48
1,Female,NATIONAL,Long Jump,5.7,5.7,5.586,5.5005,5.415,5.13
2,Male,NATIONAL,Triple Jump,15.2,15.2,14.895999999999999,14.668,14.44,13.68
3,Female,NATIONAL,Triple Jump,12.0,12.0,11.76,11.58,11.399999999999999,10.8
4,Male,NATIONAL,High Jump,2.07,2.07,2.0286,1.9975499999999997,1.9664999999999997,1.863
5,Female,NATIONAL,High Jump,1.65,1.65,1.617,1.59225,1.5675,1.4849999999999999
6,Male,TRAINING,Long Jump,6.8,6.8,6.664,6.561999999999999,6.46,6.12
7,Female,TRAINING,Long Jump,5.3,5.3,5.194,5.1145,5.034999999999999,4.77
8,Male,TRAINING,Triple Jump,14.5,14.5,14.209999999999999,13.9925,13.774999999999999,13.05
9,Female,TRAINING,Triple Jump,11.3,11.3,11.074,10.9045,10.735,10.170000000000002


In [852]:
athletes

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month
0,"Sim, Jarod",4.98m,Raffles Institution,15,20,U18,Long Jump,0.0,None,J727I11,...,Male,SA All Comers Meet 2,2026-02-08 00:00:00,2026,Local,2026-02-11 16:14:29.851514,2026-02-11 16:22:17.003853+00:00,3 days 16:22:17.003853,3,2
1,"Adam, Rabin Sunder Singh",4.83m,Raffles Institution,15,24,U18,Long Jump,0.0,None,R762W11,...,Male,SA All Comers Meet 2,2026-02-08 00:00:00,2026,Local,2026-02-11 16:14:29.851514,2026-02-11 16:22:17.003853+00:00,3 days 16:22:17.003853,3,2
2,"Chua, Hoon Kai",4.61m,Raffles Institution,15,34,U18,Long Jump,0.0,None,H269F11,...,Male,SA All Comers Meet 2,2026-02-08 00:00:00,2026,Local,2026-02-11 16:14:29.851514,2026-02-11 16:22:17.003853+00:00,3 days 16:22:17.003853,3,2
3,"NEALFAIZ ADAM, MOHAMAD NAZRUL NAZIB",4.73m,Beecity Athletics Academy,17,27,U18,Long Jump,0.0,None,M209309,...,Male,SA All Comers Meet 2,2026-02-08 00:00:00,2026,Local,2026-02-11 16:14:29.851514,2026-02-11 16:22:17.003853+00:00,3 days 16:22:17.003853,3,2
4,"TANG SONG YI, Zachary",5.25m,St Joseph's Institution,15,9,U18,Long Jump,0.0,None,None,...,Male,SA All Comers Meet 2,2026-02-08 00:00:00,2026,Local,2026-02-11 16:14:29.851514,2026-02-11 16:22:17.003853+00:00,3 days 16:22:17.003853,3,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21556,"CHENG, ZIXUAN",1.40m,Catholic High School,14,5,Novice,High Jump,0.0,None,Z573D12,...,Male,SA All Comers Meet 1,2026-02-01 00:00:00,2026,Local,2026-02-07 13:23:03.305409,2026-02-11 16:22:17.003853+00:00,10 days 16:22:17.003853,10,2
21557,"Ho, Yi Teng",1.46m,Hwa Chong Institution,14,2,Novice,High Jump,0.0,None,D372C12,...,Male,SA All Comers Meet 1,2026-02-01 00:00:00,2026,Local,2026-02-07 13:23:03.305409,2026-02-11 16:22:17.003853+00:00,10 days 16:22:17.003853,10,2
21558,"Boon, Marius",1.40m,Hwa Chong Institution,14,5,Novice,High Jump,0.0,None,Z430G12,...,Male,SA All Comers Meet 1,2026-02-01 00:00:00,2026,Local,2026-02-07 13:23:03.305409,2026-02-11 16:22:17.003853+00:00,10 days 16:22:17.003853,10,2
21559,"Tan, Ashton",1.80m,Hwa Chong Alumni Association,21,1,Advance,High Jump,0.0,None,A401I05,...,Male,SA All Comers Meet 1,2026-02-01 00:00:00,2026,Local,2026-02-07 13:23:03.305409,2026-02-11 16:22:17.003853+00:00,10 days 16:22:17.003853,10,2


In [853]:
# Select NATIONAL or TRAINING benchmarks

benchmarks = benchmarks[benchmarks['SQUAD']=='TRAINING']

In [854]:
benchmarks

,GENDER,SQUAD,EVENT,BENCHMARK,Metric,2%,3.5%,5%,10%
6,Male,TRAINING,Long Jump,6.8,6.8,6.664,6.561999999999999,6.46,6.12
7,Female,TRAINING,Long Jump,5.3,5.3,5.194,5.1145,5.034999999999999,4.77
8,Male,TRAINING,Triple Jump,14.5,14.5,14.209999999999999,13.9925,13.774999999999999,13.05
9,Female,TRAINING,Triple Jump,11.3,11.3,11.074,10.9045,10.735,10.170000000000002
10,Male,TRAINING,High Jump,1.95,1.95,1.911,1.8817499999999998,1.8524999999999998,1.755
11,Female,TRAINING,High Jump,1.57,1.57,1.5386,1.51505,1.4915,1.413


In [855]:
# Merge benchmarks onto athletes on MAPPED_EVENT and GENDER

df = pd.merge(
    left=athletes, 
    right=benchmarks,
    how='left',
    left_on=['EVENT', 'GENDER'],
    right_on=['EVENT', 'GENDER'],
)

In [856]:
df

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,delta_time,delta_time_conv,event_month,SQUAD,BENCHMARK,Metric,2%,3.5%,5%,10%
0,"Sim, Jarod",4.98m,Raffles Institution,15,20,U18,Long Jump,0.0,None,J727I11,...,3 days 16:22:17.003853,3,2,TRAINING,6.8,6.8,6.664,6.561999999999999,6.46,6.12
1,"Adam, Rabin Sunder Singh",4.83m,Raffles Institution,15,24,U18,Long Jump,0.0,None,R762W11,...,3 days 16:22:17.003853,3,2,TRAINING,6.8,6.8,6.664,6.561999999999999,6.46,6.12
2,"Chua, Hoon Kai",4.61m,Raffles Institution,15,34,U18,Long Jump,0.0,None,H269F11,...,3 days 16:22:17.003853,3,2,TRAINING,6.8,6.8,6.664,6.561999999999999,6.46,6.12
3,"NEALFAIZ ADAM, MOHAMAD NAZRUL NAZIB",4.73m,Beecity Athletics Academy,17,27,U18,Long Jump,0.0,None,M209309,...,3 days 16:22:17.003853,3,2,TRAINING,6.8,6.8,6.664,6.561999999999999,6.46,6.12
4,"TANG SONG YI, Zachary",5.25m,St Joseph's Institution,15,9,U18,Long Jump,0.0,None,None,...,3 days 16:22:17.003853,3,2,TRAINING,6.8,6.8,6.664,6.561999999999999,6.46,6.12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3545,"CHENG, ZIXUAN",1.40m,Catholic High School,14,5,Novice,High Jump,0.0,None,Z573D12,...,10 days 16:22:17.003853,10,2,TRAINING,1.95,1.95,1.911,1.8817499999999998,1.8524999999999998,1.755
3546,"Ho, Yi Teng",1.46m,Hwa Chong Institution,14,2,Novice,High Jump,0.0,None,D372C12,...,10 days 16:22:17.003853,10,2,TRAINING,1.95,1.95,1.911,1.8817499999999998,1.8524999999999998,1.755
3547,"Boon, Marius",1.40m,Hwa Chong Institution,14,5,Novice,High Jump,0.0,None,Z430G12,...,10 days 16:22:17.003853,10,2,TRAINING,1.95,1.95,1.911,1.8817499999999998,1.8524999999999998,1.755
3548,"Tan, Ashton",1.80m,Hwa Chong Alumni Association,21,1,Advance,High Jump,0.0,None,A401I05,...,10 days 16:22:17.003853,10,2,TRAINING,1.95,1.95,1.911,1.8817499999999998,1.8524999999999998,1.755


In [857]:
# replace '-' with NaN

#df['RESULT'] = df['RESULT'].replace(regex=r'–', value=np.NaN)


In [858]:
df

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,delta_time,delta_time_conv,event_month,SQUAD,BENCHMARK,Metric,2%,3.5%,5%,10%
0,"Sim, Jarod",4.98m,Raffles Institution,15,20,U18,Long Jump,0.0,None,J727I11,...,3 days 16:22:17.003853,3,2,TRAINING,6.8,6.8,6.664,6.561999999999999,6.46,6.12
1,"Adam, Rabin Sunder Singh",4.83m,Raffles Institution,15,24,U18,Long Jump,0.0,None,R762W11,...,3 days 16:22:17.003853,3,2,TRAINING,6.8,6.8,6.664,6.561999999999999,6.46,6.12
2,"Chua, Hoon Kai",4.61m,Raffles Institution,15,34,U18,Long Jump,0.0,None,H269F11,...,3 days 16:22:17.003853,3,2,TRAINING,6.8,6.8,6.664,6.561999999999999,6.46,6.12
3,"NEALFAIZ ADAM, MOHAMAD NAZRUL NAZIB",4.73m,Beecity Athletics Academy,17,27,U18,Long Jump,0.0,None,M209309,...,3 days 16:22:17.003853,3,2,TRAINING,6.8,6.8,6.664,6.561999999999999,6.46,6.12
4,"TANG SONG YI, Zachary",5.25m,St Joseph's Institution,15,9,U18,Long Jump,0.0,None,None,...,3 days 16:22:17.003853,3,2,TRAINING,6.8,6.8,6.664,6.561999999999999,6.46,6.12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3545,"CHENG, ZIXUAN",1.40m,Catholic High School,14,5,Novice,High Jump,0.0,None,Z573D12,...,10 days 16:22:17.003853,10,2,TRAINING,1.95,1.95,1.911,1.8817499999999998,1.8524999999999998,1.755
3546,"Ho, Yi Teng",1.46m,Hwa Chong Institution,14,2,Novice,High Jump,0.0,None,D372C12,...,10 days 16:22:17.003853,10,2,TRAINING,1.95,1.95,1.911,1.8817499999999998,1.8524999999999998,1.755
3547,"Boon, Marius",1.40m,Hwa Chong Institution,14,5,Novice,High Jump,0.0,None,Z430G12,...,10 days 16:22:17.003853,10,2,TRAINING,1.95,1.95,1.911,1.8817499999999998,1.8524999999999998,1.755
3548,"Tan, Ashton",1.80m,Hwa Chong Alumni Association,21,1,Advance,High Jump,0.0,None,A401I05,...,10 days 16:22:17.003853,10,2,TRAINING,1.95,1.95,1.911,1.8817499999999998,1.8524999999999998,1.755


In [859]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Jumps/')


df.to_csv('jumps_postmap_training.csv', sep=',', encoding='utf-8-sig', index=False)


In [860]:
# Convert results and seed into seconds format for mapped events only (vectorised version)

# First, normalize data as you did (remove control chars etc.)
for col in df.columns:
    df[col] = df[col].astype(str)
    df[col] = df[col].str.replace('\xa0', ' ', regex=True)
    df[col] = df[col].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
    df[col] = df[col].str.replace('\r', ' ', regex=True)
    df[col] = df[col].str.replace('\n', ' ', regex=True)
    df[col] = df[col].str.strip()

# Define a filter for rows with convertible results
invalid_results = {'—', 'None', 'DQ', 'SCR', 'FS', 'DNQ', 'DNS', 'NH', 'NM', 'FOUL', 'DNF', 'SR'}

# Apply conversion vectorized using apply, skipping invalid values
def convert_for_row(row):
    if row['RESULT'] in invalid_results:
        return ''
    return convert_time_refactored(row.name, row['EVENT'], row['RESULT'])

df['RESULT_CONV'] = df.apply(convert_for_row, axis=1)


In [861]:
# Change to numeric

df[['2%', '3.5%', '5%', '10%', 'RESULT_CONV', 'BENCHMARK']] = df[['2%', '3.5%', '5%', '10%', 'RESULT_CONV', 'BENCHMARK']].apply(pd.to_numeric, errors='coerce')

In [862]:
mask = df['CATEGORY_EVENT'].str.lower().str.contains(r'jump|throw|decathlon|heptathlon', na=True)

df.loc[mask, 'Delta2'] = df['RESULT_CONV']-df['2%']
df.loc[mask, 'Delta3.5'] = df['RESULT_CONV']-df['3.5%']
df.loc[mask, 'Delta5'] = df['RESULT_CONV']-df['5%']
df.loc[mask, 'Delta10'] = df['RESULT_CONV']-df['10%']
df.loc[mask, 'Delta_Benchmark'] = df['RESULT_CONV']-df['BENCHMARK']

df.loc[~mask, 'Delta2'] =  df['2%'] - df['RESULT_CONV']
df.loc[~mask, 'Delta3.5'] = df['3.5%'] - df['RESULT_CONV']
df.loc[~mask, 'Delta5'] = df['5%'] - df['RESULT_CONV']
df.loc[~mask, 'Delta10'] = df['10%'] - df['RESULT_CONV']
df.loc[~mask, 'Delta_Benchmark'] = df['BENCHMARK'] - df['RESULT_CONV']



In [863]:
# Performance metric to filter out athletes

df['PERF_SCALAR']=df['Delta5']/df['BENCHMARK']*100

In [864]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Jumps/')


df.to_csv('jumps_postmap_benchmarked_training.csv', sep=',', encoding='utf-8-sig', index=False)


In [865]:
df

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,3.5%,5%,10%,RESULT_CONV,Delta2,Delta3.5,Delta5,Delta10,Delta_Benchmark,PERF_SCALAR
0,"Sim, Jarod",4.98m,Raffles Institution,15,20,U18,Long Jump,0.0,None,J727I11,...,6.56200,6.4600,6.120,4.98,-1.684,-1.58200,-1.4800,-1.140,-1.82,-21.764706
1,"Adam, Rabin Sunder Singh",4.83m,Raffles Institution,15,24,U18,Long Jump,0.0,None,R762W11,...,6.56200,6.4600,6.120,4.83,-1.834,-1.73200,-1.6300,-1.290,-1.97,-23.970588
2,"Chua, Hoon Kai",4.61m,Raffles Institution,15,34,U18,Long Jump,0.0,None,H269F11,...,6.56200,6.4600,6.120,4.61,-2.054,-1.95200,-1.8500,-1.510,-2.19,-27.205882
3,"NEALFAIZ ADAM, MOHAMAD NAZRUL NAZIB",4.73m,Beecity Athletics Academy,17,27,U18,Long Jump,0.0,None,M209309,...,6.56200,6.4600,6.120,4.73,-1.934,-1.83200,-1.7300,-1.390,-2.07,-25.441176
4,"TANG SONG YI, Zachary",5.25m,St Joseph's Institution,15,9,U18,Long Jump,0.0,None,None,...,6.56200,6.4600,6.120,5.25,-1.414,-1.31200,-1.2100,-0.870,-1.55,-17.794118
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3545,"CHENG, ZIXUAN",1.40m,Catholic High School,14,5,Novice,High Jump,0.0,None,Z573D12,...,1.88175,1.8525,1.755,1.40,-0.511,-0.48175,-0.4525,-0.355,-0.55,-23.205128
3546,"Ho, Yi Teng",1.46m,Hwa Chong Institution,14,2,Novice,High Jump,0.0,None,D372C12,...,1.88175,1.8525,1.755,1.46,-0.451,-0.42175,-0.3925,-0.295,-0.49,-20.128205
3547,"Boon, Marius",1.40m,Hwa Chong Institution,14,5,Novice,High Jump,0.0,None,Z430G12,...,1.88175,1.8525,1.755,1.40,-0.511,-0.48175,-0.4525,-0.355,-0.55,-23.205128
3548,"Tan, Ashton",1.80m,Hwa Chong Alumni Association,21,1,Advance,High Jump,0.0,None,A401I05,...,1.88175,1.8525,1.755,1.80,-0.111,-0.08175,-0.0525,0.045,-0.15,-2.692308


In [866]:

# Read a variation name list and corrections from CSVs
'''
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/OCTC/')

names = pd.read_csv("name_variations.csv")

for index, row in names.iterrows():
        
    print(names.VARIATION, names.NAME)
    df['NAME'] = df['NAME'].replace(regex=rf"{row['VARIATION']}", value=f"{row['NAME']}")
'''

'\nos.chdir(\'/Users/veesheenyuen/Desktop/DataScience/SAA/OCTC/\')\n\nnames = pd.read_csv("name_variations.csv")\n\nfor index, row in names.iterrows():\n        \n    print(names.VARIATION, names.NAME)\n    df[\'NAME\'] = df[\'NAME\'].replace(regex=rf"{row[\'VARIATION\']}", value=f"{row[\'NAME\']}")\n'

In [867]:
# Fastest execution speed version
# Normalize function as before

def normalize_text(s):
    return (str(s)
            .replace('\xa0', '')
            .replace('\r', '')
            .replace('\n', '')
            .strip()
            .casefold())

# Normalize dataframe
df['NAME'] = df['NAME'].apply(normalize_text)

# Load variations file and normalize
file_path = "gs://name_variations/name_variations.csv"
names = pd.read_csv(file_path,
                    sep=',',
                    storage_options={"token": '/Users/veesheenyuen/Desktop/DataScience/Keys/saa-analytics-7c8937b70609.json'})

names['VARIATION'] = names['VARIATION'].apply(normalize_text)
names['NAME'] = names['NAME'].apply(normalize_text)

# Precompile all regex patterns safely

compiled_patterns = []
for pattern_str, replacement in zip(names['VARIATION'], names['NAME']):
    try:
        compiled_re = re.compile(pattern_str)
        compiled_patterns.append( (compiled_re, replacement) )
    except re.error as e:
        print(f"Skipping invalid regex pattern: {pattern_str} Error: {e}")

# Iterate over all patterns and apply replacements using precompiled regexes
for regex, replacement in compiled_patterns:
    df['NAME'] = df['NAME'].str.replace(regex, replacement, regex=True)

# Capitalize final standardized names
df['NAME'] = df['NAME'].str.title()


In [868]:
compiled_patterns

[(re.compile(r'soh, aidan michael zi ren', re.UNICODE),
  'aidan michael soh zi ren'),
 (re.compile(r'aidan michael soh zi ren', re.UNICODE),
  'aidan michael soh zi ren'),
 (re.compile(r'soh zi ren, aidan michael', re.UNICODE),
  'aidan michael soh zi ren'),
 (re.compile(r'soh zi ren, aidan michael', re.UNICODE),
  'aidan michael soh zi ren'),
 (re.compile(r'soh, aidan michael', re.UNICODE), 'aidan michael soh zi ren'),
 (re.compile(r'., aidan michael soh zi', re.UNICODE),
  'aidan michael soh zi ren'),
 (re.compile(r'^aidan michael soh$', re.UNICODE), 'aidan michael soh zi ren'),
 (re.compile(r'ang, chen xiang', re.UNICODE), 'ang chen xiang'),
 (re.compile(r'cheng xiang ang', re.UNICODE), 'ang chen xiang'),
 (re.compile(r'chen, xiang', re.UNICODE), 'ang chen xiang'),
 (re.compile(r'chen xiang', re.UNICODE), 'ang chen xiang'),
 (re.compile(r'^ang, chen xiang$', re.UNICODE), 'ang chen xiang'),
 (re.compile(r'^chen xiang ang$', re.UNICODE), 'ang chen xiang'),
 (re.compile(r'., ang chen 

In [869]:
df

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,3.5%,5%,10%,RESULT_CONV,Delta2,Delta3.5,Delta5,Delta10,Delta_Benchmark,PERF_SCALAR
0,"Sim, Jarod",4.98m,Raffles Institution,15,20,U18,Long Jump,0.0,None,J727I11,...,6.56200,6.4600,6.120,4.98,-1.684,-1.58200,-1.4800,-1.140,-1.82,-21.764706
1,"Adam, Rabin Sunder Singh",4.83m,Raffles Institution,15,24,U18,Long Jump,0.0,None,R762W11,...,6.56200,6.4600,6.120,4.83,-1.834,-1.73200,-1.6300,-1.290,-1.97,-23.970588
2,"Chua, Hoon Kai",4.61m,Raffles Institution,15,34,U18,Long Jump,0.0,None,H269F11,...,6.56200,6.4600,6.120,4.61,-2.054,-1.95200,-1.8500,-1.510,-2.19,-27.205882
3,"Nealfaiz Adam, Mohamad Nazrul Nazib",4.73m,Beecity Athletics Academy,17,27,U18,Long Jump,0.0,None,M209309,...,6.56200,6.4600,6.120,4.73,-1.934,-1.83200,-1.7300,-1.390,-2.07,-25.441176
4,"Tang Song Yi, Zachary",5.25m,St Joseph's Institution,15,9,U18,Long Jump,0.0,None,None,...,6.56200,6.4600,6.120,5.25,-1.414,-1.31200,-1.2100,-0.870,-1.55,-17.794118
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3545,"Cheng, Zixuan",1.40m,Catholic High School,14,5,Novice,High Jump,0.0,None,Z573D12,...,1.88175,1.8525,1.755,1.40,-0.511,-0.48175,-0.4525,-0.355,-0.55,-23.205128
3546,"Ho, Yi Teng",1.46m,Hwa Chong Institution,14,2,Novice,High Jump,0.0,None,D372C12,...,1.88175,1.8525,1.755,1.46,-0.451,-0.42175,-0.3925,-0.295,-0.49,-20.128205
3547,"Boon, Marius",1.40m,Hwa Chong Institution,14,5,Novice,High Jump,0.0,None,Z430G12,...,1.88175,1.8525,1.755,1.40,-0.511,-0.48175,-0.4525,-0.355,-0.55,-23.205128
3548,"Tan, Ashton",1.80m,Hwa Chong Alumni Association,21,1,Advance,High Jump,0.0,None,A401I05,...,1.88175,1.8525,1.755,1.80,-0.111,-0.08175,-0.0525,0.045,-0.15,-2.692308


In [870]:
# Exclude foreigners from MALAYSIA, THAILAND etc.

#df_select = df[(df['TEAM']!='Malaysia') & (df['TEAM']!='THAILAND') & (df['TEAM']!='China') & (df['TEAM']!='South Korea') & (df['TEAM']!='Laos') & (df['TEAM']!='Philippines') & (df['TEAM']!='Piboonbumpen Thailand') & (df['TEAM']!='Chinese Taipei') & (df['TEAM']!='Gurkha Contingent') & (df['TEAM']!='Australia') & (df['TEAM']!='Piboonbumpen Thailand') & (df['TEAM']!='Hong Kong') & (df['TEAM']!='PERAK')] 

df_select = df[(df['TEAM']!='Malaysia')&(df['TEAM']!='THAILAND')&(df['TEAM']!='China')&(df['TEAM']!='Thailand') 
                       &(df['TEAM']!='South Korea')&(df['TEAM']!='Laos')&(df['TEAM']!='Myanmar') 
                       &(df['TEAM']!='Philippines')&(df['TEAM']!='Piboonbumpen Thailand') 
                       &(df['TEAM']!='Chinese Taipei')&(df['TEAM']!='Gurkha Contingent') 
                       &(df['TEAM']!='Australia')&(df['TEAM']!='Piboonbumpen Thailand') 
                       &(df['TEAM']!='Hong Kong')&(df['TEAM']!='PERAK')&(df['TEAM']!='Sri Lanka') 
                       &(df['TEAM']!='Indonesia')&(df['TEAM']!='THAILAND')&(df['TEAM']!='MALAYSIA') 
                       &(df['TEAM']!='PHILIPPINES') & (df['TEAM']!='SOUTH KOREA')&(df['TEAM']!='Waseda') 
                       &(df['TEAM']!='LAOS')&(df['TEAM']!='CHINESE TAIPEI')&(df['TEAM']!='Vietnam')
                       &(df['TEAM']!='INDIA')&(df['TEAM']!='Hong Kong, China')&(df['TEAM']!='AIC JAPAN')
                       &(df['NATIONALITY']!='GBR')&(df['NATIONALITY']!='JPN')&(df['NATIONALITY']!='SRI')&(df['NATIONALITY']!='SAM')
                       &(df['NATIONALITY']!='THA')&(df['NATIONALITY']!='IND')] 

In [871]:
df_select

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,3.5%,5%,10%,RESULT_CONV,Delta2,Delta3.5,Delta5,Delta10,Delta_Benchmark,PERF_SCALAR
0,"Sim, Jarod",4.98m,Raffles Institution,15,20,U18,Long Jump,0.0,None,J727I11,...,6.56200,6.4600,6.120,4.98,-1.684,-1.58200,-1.4800,-1.140,-1.82,-21.764706
1,"Adam, Rabin Sunder Singh",4.83m,Raffles Institution,15,24,U18,Long Jump,0.0,None,R762W11,...,6.56200,6.4600,6.120,4.83,-1.834,-1.73200,-1.6300,-1.290,-1.97,-23.970588
2,"Chua, Hoon Kai",4.61m,Raffles Institution,15,34,U18,Long Jump,0.0,None,H269F11,...,6.56200,6.4600,6.120,4.61,-2.054,-1.95200,-1.8500,-1.510,-2.19,-27.205882
3,"Nealfaiz Adam, Mohamad Nazrul Nazib",4.73m,Beecity Athletics Academy,17,27,U18,Long Jump,0.0,None,M209309,...,6.56200,6.4600,6.120,4.73,-1.934,-1.83200,-1.7300,-1.390,-2.07,-25.441176
4,"Tang Song Yi, Zachary",5.25m,St Joseph's Institution,15,9,U18,Long Jump,0.0,None,None,...,6.56200,6.4600,6.120,5.25,-1.414,-1.31200,-1.2100,-0.870,-1.55,-17.794118
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3545,"Cheng, Zixuan",1.40m,Catholic High School,14,5,Novice,High Jump,0.0,None,Z573D12,...,1.88175,1.8525,1.755,1.40,-0.511,-0.48175,-0.4525,-0.355,-0.55,-23.205128
3546,"Ho, Yi Teng",1.46m,Hwa Chong Institution,14,2,Novice,High Jump,0.0,None,D372C12,...,1.88175,1.8525,1.755,1.46,-0.451,-0.42175,-0.3925,-0.295,-0.49,-20.128205
3547,"Boon, Marius",1.40m,Hwa Chong Institution,14,5,Novice,High Jump,0.0,None,Z430G12,...,1.88175,1.8525,1.755,1.40,-0.511,-0.48175,-0.4525,-0.355,-0.55,-23.205128
3548,"Tan, Ashton",1.80m,Hwa Chong Alumni Association,21,1,Advance,High Jump,0.0,None,A401I05,...,1.88175,1.8525,1.755,1.80,-0.111,-0.08175,-0.0525,0.045,-0.15,-2.692308


In [872]:
df_PV = df_select[(df_select['EVENT']=='Pole Vault') & (df_select['GENDER']=='Female')]

In [873]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Jumps/')

df_PV.to_csv('check.csv', encoding='utf-8')

In [749]:
'''
# Read list of foreigners from GCS bucket

file_path = "gs://name_lists/List of Foreigners.csv"
foreigners = pd.read_csv(file_path,
                 sep=",",
                 encoding="unicode escape",
                 storage_options={"token": '/Users/veesheenyuen/Desktop/DataScience/Keys/saa-analytics-7c8937b70609.json'})
'''

'\n# Read list of foreigners from GCS bucket\n\nfile_path = "gs://name_lists/List of Foreigners.csv"\nforeigners = pd.read_csv(file_path,\n                 sep=",",\n                 encoding="unicode escape",\n                 storage_options={"token": \'/Users/veesheenyuen/Desktop/DataScience/Keys/saa-analytics-7c8937b70609.json\'})\n'

In [750]:
#foreigners

In [751]:
#df_select['NAME'].str.casefold()

In [752]:
'''
foreigners['V1'] = foreigners['LAST_NAME']+' '+foreigners['FIRST_NAME']
foreigners['V2'] = foreigners['FIRST_NAME']+' '+foreigners['LAST_NAME']
foreigners['V3'] = foreigners['LAST_NAME']+', '+foreigners['FIRST_NAME']
foreigners['V4'] = foreigners['FIRST_NAME']+' '+foreigners['LAST_NAME']

for1 = foreigners['V1'].dropna().tolist()
for2 = foreigners['V2'].dropna().tolist()
for3 = foreigners['V3'].dropna().tolist()
for4 = foreigners['V4'].dropna().tolist()

foreign_list = for1+for2+for3+for4 

foreign_list_casefold=[s.casefold() for s in foreign_list]

exclusions = foreign_list_casefold

no_foreigners_list = df_select.loc[~df['NAME'].str.casefold().isin(exclusions)]  # ~ means NOT IN. DROP spex carded athletes

'''

"\nforeigners['V1'] = foreigners['LAST_NAME']+' '+foreigners['FIRST_NAME']\nforeigners['V2'] = foreigners['FIRST_NAME']+' '+foreigners['LAST_NAME']\nforeigners['V3'] = foreigners['LAST_NAME']+', '+foreigners['FIRST_NAME']\nforeigners['V4'] = foreigners['FIRST_NAME']+' '+foreigners['LAST_NAME']\n\nfor1 = foreigners['V1'].dropna().tolist()\nfor2 = foreigners['V2'].dropna().tolist()\nfor3 = foreigners['V3'].dropna().tolist()\nfor4 = foreigners['V4'].dropna().tolist()\n\nforeign_list = for1+for2+for3+for4 \n\nforeign_list_casefold=[s.casefold() for s in foreign_list]\n\nexclusions = foreign_list_casefold\n\nno_foreigners_list = df_select.loc[~df['NAME'].str.casefold().isin(exclusions)]  # ~ means NOT IN. DROP spex carded athletes\n\n"

In [753]:
# Choose best performance per event and athlete (robust version)

# 1) Convert to numeric FIRST
df_select["PERF_SCALAR"] = pd.to_numeric(df_select["PERF_SCALAR"], errors="coerce")

# 2) Drop non-numeric (NaNs) and non-finite values
df2 = df_select[np.isfinite(df_select["PERF_SCALAR"])].copy()

# 3) Normalize grouping keys to avoid silent split by stray spaces
for col in ["NAME", "EVENT"]:
    df2[col] = df2[col].astype(str).str.strip()

# 4) Keep the best (largest) per group
top_performers_clean = (
    df2.sort_values(
        ["EVENT", "NAME", "PERF_SCALAR"],
        ascending=[True, True, False]
    )
    .drop_duplicates(subset=["EVENT", "NAME"], keep="first")
    .reset_index(drop=True)
)

top_performers_clean

/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_11839/547807012.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_select["PERF_SCALAR"] = pd.to_numeric(df_select["PERF_SCALAR"], errors="coerce")


,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,3.5%,5%,10%,RESULT_CONV,Delta2,Delta3.5,Delta5,Delta10,Delta_Benchmark,PERF_SCALAR
0,"., Misha Pandey",1.35m,Team Start Singapore,14,2,Novice,High Jump,0.0,None,M261B12,...,1.51505,1.4915,1.413,1.35,-0.1886,-0.16505,-0.1415,-0.063,-0.22,-9.012739
1,"Abe, Jason",1.76m,VICTORIA SCHOOL,16,1,Novice,High Jump,0.0,None,J638B09,...,1.88175,1.8525,1.755,1.76,-0.1510,-0.12175,-0.0925,0.005,-0.19,-4.743590
2,"Adhyatma Manika, I Putu Gede",1.95m,Bali,20.0,10,Open,High Jump,0.0,None,I334905,...,1.88175,1.8525,1.755,1.95,0.0390,0.06825,0.0975,0.195,0.00,5.000000
3,Aidan Chua Kiat Oon,1.4,HCI,,11.0,C,High Jump,,,,...,1.88175,1.8525,1.755,1.40,-0.5110,-0.48175,-0.4525,-0.355,-0.55,-23.205128
4,Alex Joseph Lam,1.61,SJI,,4.0,C,High Jump,,,,...,1.88175,1.8525,1.755,1.61,-0.3010,-0.27175,-0.2425,-0.145,-0.34,-12.435897
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1715,"Zhao, Daniel",11.67m,Hwa Chong Institution,14,3,U15,Triple Jump,0.0,None,None,...,13.99250,13.7750,13.050,11.67,-2.5400,-2.32250,-2.1050,-1.380,-2.83,-14.517241
1716,Zheng Justin De,10.47m,National Junior College,14,12,U15,Triple Jump,0.0,None,J034B11,...,13.99250,13.7750,13.050,10.47,-3.7400,-3.52250,-3.3050,-2.580,-4.03,-22.793103
1717,Zhong Chuhan,12.26,,,,,Triple Jump,,,,...,10.90450,10.7350,10.170,12.26,1.1860,1.35550,1.5250,2.090,0.96,13.495575
1718,Zhou Xuanyu,9.12,DHS,,10.0,C,Triple Jump,,,,...,10.90450,10.7350,10.170,9.12,-1.9540,-1.78450,-1.6150,-1.050,-2.18,-14.292035


In [754]:
top_performers_clean.reset_index(inplace=True)


In [755]:
top_performers_clean

,index,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,...,3.5%,5%,10%,RESULT_CONV,Delta2,Delta3.5,Delta5,Delta10,Delta_Benchmark,PERF_SCALAR
0,0,"., Misha Pandey",1.35m,Team Start Singapore,14,2,Novice,High Jump,0.0,None,...,1.51505,1.4915,1.413,1.35,-0.1886,-0.16505,-0.1415,-0.063,-0.22,-9.012739
1,1,"Abe, Jason",1.76m,VICTORIA SCHOOL,16,1,Novice,High Jump,0.0,None,...,1.88175,1.8525,1.755,1.76,-0.1510,-0.12175,-0.0925,0.005,-0.19,-4.743590
2,2,"Adhyatma Manika, I Putu Gede",1.95m,Bali,20.0,10,Open,High Jump,0.0,None,...,1.88175,1.8525,1.755,1.95,0.0390,0.06825,0.0975,0.195,0.00,5.000000
3,3,Aidan Chua Kiat Oon,1.4,HCI,,11.0,C,High Jump,,,...,1.88175,1.8525,1.755,1.40,-0.5110,-0.48175,-0.4525,-0.355,-0.55,-23.205128
4,4,Alex Joseph Lam,1.61,SJI,,4.0,C,High Jump,,,...,1.88175,1.8525,1.755,1.61,-0.3010,-0.27175,-0.2425,-0.145,-0.34,-12.435897
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1715,1715,"Zhao, Daniel",11.67m,Hwa Chong Institution,14,3,U15,Triple Jump,0.0,None,...,13.99250,13.7750,13.050,11.67,-2.5400,-2.32250,-2.1050,-1.380,-2.83,-14.517241
1716,1716,Zheng Justin De,10.47m,National Junior College,14,12,U15,Triple Jump,0.0,None,...,13.99250,13.7750,13.050,10.47,-3.7400,-3.52250,-3.3050,-2.580,-4.03,-22.793103
1717,1717,Zhong Chuhan,12.26,,,,,Triple Jump,,,...,10.90450,10.7350,10.170,12.26,1.1860,1.35550,1.5250,2.090,0.96,13.495575
1718,1718,Zhou Xuanyu,9.12,DHS,,10.0,C,Triple Jump,,,...,10.90450,10.7350,10.170,9.12,-1.9540,-1.78450,-1.6150,-1.050,-2.18,-14.292035


In [689]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Jumps/')

top_performers_clean.to_csv('jumps_top_performers_training.csv', encoding='utf-8')

In [690]:
# Choose best performance for each event

#tiered_performers = top_performers_clean.sort_values(['GENDER', 'MAPPED_EVENT', 'PERF_SCALAR'],ascending=False).groupby(['MAPPED_EVENT', 'NAME']).head(1)

tiered_performers = top_performers_clean


In [691]:
tiered_performers

,index,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,...,3.5%,5%,10%,RESULT_CONV,Delta2,Delta3.5,Delta5,Delta10,Delta_Benchmark,PERF_SCALAR
0,0,"., Misha Pandey",1.35m,Team Start Singapore,14,2,Novice,High Jump,0.0,None,...,1.51505,1.4915,1.413,1.35,-0.1886,-0.16505,-0.1415,-0.063,-0.22,-9.012739
1,1,"Abe, Jason",1.76m,VICTORIA SCHOOL,16,1,Novice,High Jump,0.0,None,...,1.88175,1.8525,1.755,1.76,-0.1510,-0.12175,-0.0925,0.005,-0.19,-4.743590
2,2,"Adhyatma Manika, I Putu Gede",1.95m,Bali,20.0,10,Open,High Jump,0.0,None,...,1.88175,1.8525,1.755,1.95,0.0390,0.06825,0.0975,0.195,0.00,5.000000
3,3,Aidan Chua Kiat Oon,1.4,HCI,,11.0,C,High Jump,,,...,1.88175,1.8525,1.755,1.40,-0.5110,-0.48175,-0.4525,-0.355,-0.55,-23.205128
4,4,Alex Joseph Lam,1.61,SJI,,4.0,C,High Jump,,,...,1.88175,1.8525,1.755,1.61,-0.3010,-0.27175,-0.2425,-0.145,-0.34,-12.435897
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1715,1715,"Zhao, Daniel",11.67m,Hwa Chong Institution,14,3,U15,Triple Jump,0.0,None,...,13.99250,13.7750,13.050,11.67,-2.5400,-2.32250,-2.1050,-1.380,-2.83,-14.517241
1716,1716,Zheng Justin De,10.47m,National Junior College,14,12,U15,Triple Jump,0.0,None,...,13.99250,13.7750,13.050,10.47,-3.7400,-3.52250,-3.3050,-2.580,-4.03,-22.793103
1717,1717,Zhong Chuhan,12.26,,,,,Triple Jump,,,...,10.90450,10.7350,10.170,12.26,1.1860,1.35550,1.5250,2.090,0.96,13.495575
1718,1718,Zhou Xuanyu,9.12,DHS,,10.0,C,Triple Jump,,,...,10.90450,10.7350,10.170,9.12,-1.9540,-1.78450,-1.6150,-1.050,-2.18,-14.292035


In [692]:
# Identify Tier 1/2/3 performers

#top_performers_clean['TIER'] = np.where((top_performers_clean['Delta_Benchmark']>=0), 'Tier 1',    
#                                np.where(((top_performers_clean['Delta_Benchmark']<0) & (top_performers_clean['Delta2']>=0)), 'Tier2',
#                                np.where(((top_performers_clean['Delta2']<0) & (top_performers_clean['Delta3.5']>=0)), 'Tier3', ' ')))


tiered_performers['TIER'] = np.where((tiered_performers['Delta_Benchmark']>=0), 'Tier 1',    
                                np.where(((tiered_performers['Delta_Benchmark']<0) & (tiered_performers['Delta2']>=0)), 'Tier 2',
                                np.where(((tiered_performers['Delta2']<0) & (tiered_performers['Delta3.5']>=0)), 'Tier 3',
                                np.where(((tiered_performers['Delta3.5']<0) & (tiered_performers['Delta5']>=0)), 'Tier 4',
                                np.where(((tiered_performers['Delta5']<0) & (tiered_performers['Delta10']>=0)), 'Tier 5', ' ')))))



In [693]:
# Drop rows without a benchmark

final_df = tiered_performers[tiered_performers['BENCHMARK'].notna()]


In [694]:
final_df

,index,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,...,5%,10%,RESULT_CONV,Delta2,Delta3.5,Delta5,Delta10,Delta_Benchmark,PERF_SCALAR,TIER
0,0,"., Misha Pandey",1.35m,Team Start Singapore,14,2,Novice,High Jump,0.0,None,...,1.4915,1.413,1.35,-0.1886,-0.16505,-0.1415,-0.063,-0.22,-9.012739,
1,1,"Abe, Jason",1.76m,VICTORIA SCHOOL,16,1,Novice,High Jump,0.0,None,...,1.8525,1.755,1.76,-0.1510,-0.12175,-0.0925,0.005,-0.19,-4.743590,Tier 5
2,2,"Adhyatma Manika, I Putu Gede",1.95m,Bali,20.0,10,Open,High Jump,0.0,None,...,1.8525,1.755,1.95,0.0390,0.06825,0.0975,0.195,0.00,5.000000,Tier 1
3,3,Aidan Chua Kiat Oon,1.4,HCI,,11.0,C,High Jump,,,...,1.8525,1.755,1.40,-0.5110,-0.48175,-0.4525,-0.355,-0.55,-23.205128,
4,4,Alex Joseph Lam,1.61,SJI,,4.0,C,High Jump,,,...,1.8525,1.755,1.61,-0.3010,-0.27175,-0.2425,-0.145,-0.34,-12.435897,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1715,1715,"Zhao, Daniel",11.67m,Hwa Chong Institution,14,3,U15,Triple Jump,0.0,None,...,13.7750,13.050,11.67,-2.5400,-2.32250,-2.1050,-1.380,-2.83,-14.517241,
1716,1716,Zheng Justin De,10.47m,National Junior College,14,12,U15,Triple Jump,0.0,None,...,13.7750,13.050,10.47,-3.7400,-3.52250,-3.3050,-2.580,-4.03,-22.793103,
1717,1717,Zhong Chuhan,12.26,,,,,Triple Jump,,,...,10.7350,10.170,12.26,1.1860,1.35550,1.5250,2.090,0.96,13.495575,Tier 1
1718,1718,Zhou Xuanyu,9.12,DHS,,10.0,C,Triple Jump,,,...,10.7350,10.170,9.12,-1.9540,-1.78450,-1.6150,-1.050,-2.18,-14.292035,


In [695]:
# Process dates to extract age

# Map NSG divisions into age

mask = (final_df['DIVISION'].str.contains(r'A', na=False))
final_df.loc[mask, 'AGE'] = '18.5'

mask = (final_df['DIVISION'].str.contains(r'B', na=False))
final_df.loc[mask, 'AGE'] = '16'

mask = (final_df['DIVISION'].str.contains(r'C', na=False))
final_df.loc[mask, 'AGE'] = '13.5'

mask = (final_df['DIVISION'].str.contains(r'O', na=False))
final_df.loc[mask, 'AGE'] = '12'



In [696]:
def length(string):

    B = ''
    year = ''

    try:

        length = len(string)

        if length == 2:

            string = '19' + string

        elif length == 1:

            string = ''

        else:

            pass

        if string is not None or len(string) != 1:

            B = parser.parse(string, dayfirst=True)
                        
    except:

        pass

    return B


final_df['DOB_new'] = final_df['DOB'].apply(length)



#B = parser.parse("10-09-2021", dayfirst = True)

In [697]:
final_df['DOB_new'] = pd.to_datetime(final_df['DOB_new'], errors='coerce')

final_df['year_extract']=final_df['DOB_new'].dt.strftime('%Y')

final_df['year_extract'] = pd.to_numeric(final_df['year_extract'])

final_df['age_extract'] = 2025 - final_df['year_extract']


In [698]:
def age(number):  # correct negative age numbers
    
    if number<0:
        
        number+=100
        
    return number


final_df['age_extract']=final_df['age_extract'].apply(age)


In [699]:
# If NSG event then choose AGE otherwise choose age_extract

condition1 = final_df['COMPETITION']=='National School Games'
#condition2=((df['CATEGORY_EVENT']=='Jump')|(df['CATEGORY_EVENT']=='Throw'))
#condition3=df['SEED_CONV']<df['RESULT_CONV']
#condition4=~((df['CATEGORY_EVENT']=='Jump')|(df['CATEGORY_EVENT']=='Throw'))


final_df['age_extract'] = final_df['AGE'].where((condition1), final_df['age_extract'].values)


In [700]:
# Change to numeric

final_df[['age_extract']] = final_df[['age_extract']].apply(pd.to_numeric)

In [701]:
final_tiered_selection = final_df[final_df['TIER']!=' ']

In [702]:
final_tiered_selection.columns

Index(['index', 'NAME', 'RESULT', 'TEAM', 'AGE', 'COMPETITION_RANK',
       'DIVISION', 'EVENT', 'DISTANCE', 'EVENT_CLASS', 'UNIQUE_ID', 'DOB',
       'NATIONALITY', 'WIND', 'CATEGORY_EVENT', 'GENDER', 'COMPETITION',
       'DATE', 'YEAR', 'REGION', 'TIMESTAMP', 'NOW', 'delta_time',
       'delta_time_conv', 'event_month', 'SQUAD', 'BENCHMARK', 'Metric', '2%',
       '3.5%', '5%', '10%', 'RESULT_CONV', 'Delta2', 'Delta3.5', 'Delta5',
       'Delta10', 'Delta_Benchmark', 'PERF_SCALAR', 'TIER', 'DOB_new',
       'year_extract', 'age_extract'],
      dtype='object')

In [703]:
final_tiered_selection = (
    final_tiered_selection.sort_values(
        ["EVENT", "GENDER", "PERF_SCALAR"],
        ascending=[True, True, False]
    )
    .reset_index(drop=True)
)



In [704]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Jumps/')

final_tiered_selection.to_csv('final_selection_training.csv', encoding='utf-8')